# Week 4 Coding Assignment: Data Acquisition — SQL, APIs & Web Data

**Data Science: Foundations to Applications (SCD-4XX) | Kyungdong University, Global Campus**

**Outcome being assessed:** Retrieve data from databases and external sources programmatically.

## Instructions
- Complete every function below by replacing `# TODO` with your code.
- **Always use `?` placeholders in SQL queries that include a variable — never an f-string or `%` formatting.** This is graded as part of the assignment.
- Run the test cell below each function to check your answer.
- When finished: save the notebook, then `git add`, `git commit`, and `git push` to your `data-science-labs-2026` repository, per the Week 1 lab.

**Parts:**
- Part A — SQL Fundamentals (Problems 1–4)
- Part B — Python + SQL Safely (Problems 5–6)
- Part C — REST APIs (Problems 7–9)
- Stretch Challenge (optional) — Ethical Web Scraping


> ⚠️ **Run this notebook in Jupyter Notebook (via Anaconda) — not Google Colab.**
This course uses your local Jupyter environment for all labs and assignments.
Open it the same way you did in Week 1: navigate to your cloned repo folder in a terminal and run `jupyter notebook`.

## Setup — run this first
This builds a sample `retail.db` SQLite database with two linked tables, `customers` and `orders`, that every problem below uses. You don't need to modify this cell.

In [1]:
import sqlite3
import random
import time
import requests
import pandas as pd
from datetime import date, timedelta

random.seed(7)

conn = sqlite3.connect("retail.db")
cur = conn.cursor()

cur.execute("DROP TABLE IF EXISTS orders")
cur.execute("DROP TABLE IF EXISTS customers")

cur.execute("""
CREATE TABLE customers (
    customer_id INTEGER PRIMARY KEY,
    name TEXT,
    city TEXT,
    segment TEXT
)
""")
cur.execute("""
CREATE TABLE orders (
    order_id INTEGER PRIMARY KEY,
    customer_id INTEGER,
    order_date TEXT,
    amount REAL,
    FOREIGN KEY (customer_id) REFERENCES customers(customer_id)
)
""")

cities = ["Seoul", "Busan", "Incheon", "Daegu", "Gwangju"]
segments = ["Retail", "Wholesale", "Online"]
names = ["Kim Minji","Park Jisoo","Lee Jaehyun","Choi Seoyeon","Jung Hyunwoo",
         "Kang Yuna","Yoon Doyoon","Han Soomin","Oh Jiwoo","Shin Taeyang",
         "Baek Narae","Song Junho","Moon Hana","Ahn Jiho","Jang Yerin"]

customers = [(i, name, random.choice(cities), random.choice(segments))
             for i, name in enumerate(names, start=1)]
cur.executemany("INSERT INTO customers VALUES (?,?,?,?)", customers)

start_date = date(2026, 1, 1)
order_id, orders = 1, []
for c in customers:
    for _ in range(random.randint(2, 8)):
        d = start_date + timedelta(days=random.randint(0, 270))
        amt = round(random.uniform(15000, 450000), -2)
        orders.append((order_id, c[0], d.isoformat(), amt))
        order_id += 1
cur.executemany("INSERT INTO orders VALUES (?,?,?,?)", orders)
conn.commit()

print(f"retail.db ready: {len(customers)} customers, {len(orders)} orders.")

retail.db ready: 15 customers, 73 orders.


---
## Part A — SQL Fundamentals

### Problem 1 — Find Customers by City and Segment
Return the `name`, `city`, and `segment` of every customer in a given `city` and `segment`.

**Hint:** `WHERE ... AND ...`

In [2]:

def customers_in_city_segment(conn, city, segment):
    """
    Find customers matching the specified city and segment.
    Return a Pandas DataFrame.
    """

    query = """
        SELECT name, city, segment
        FROM customers
        WHERE city = ?
          AND segment = ?
    """

    result = pd.read_sql(
        query,
        conn,
        params=(city, segment)
    )

    return result


# Test
print(customers_in_city_segment(conn, "Seoul", "Online"))


         name   city segment
0   Kang Yuna  Seoul  Online
1  Song Junho  Seoul  Online


### Problem 2 — Join Customers with Their Orders
Return `name`, `order_date`, and `amount` for every order, joined with the customer who placed it. Limit to the first 10 rows.

**Hint:** `INNER JOIN ... ON ...`, `LIMIT`

In [3]:

def orders_with_customer_names(conn):
    """
    Return the first 10 orders with:
    name, order_date, amount
    """

    query = """
        SELECT
            customers.name,
            orders.order_date,
            orders.amount
        FROM customers
        INNER JOIN orders
            ON customers.customer_id = orders.customer_id
        ORDER BY orders.order_id
        LIMIT 10
    """

    return pd.read_sql(query, conn)


# Assignment test
print(orders_with_customer_names(conn))


         name  order_date    amount
0   Kim Minji  2026-01-26  439700.0
1   Kim Minji  2026-01-24  257100.0
2   Kim Minji  2026-03-10  141000.0
3   Kim Minji  2026-03-15  250200.0
4   Kim Minji  2026-06-07  258700.0
5  Park Jisoo  2026-04-03   59800.0
6  Park Jisoo  2026-04-07  177000.0
7  Park Jisoo  2026-02-02  260500.0
8  Park Jisoo  2026-04-16  230900.0
9  Park Jisoo  2026-08-07  353100.0


### Problem 3 — Total Revenue per Customer (High Spenders Only)
Return each customer's `name` and their `total_spent` (sum of `amount`), but only for customers whose total exceeds a given `min_total`. Sort by `total_spent` descending.

**Hint:** `GROUP BY`, `HAVING`, `ORDER BY ... DESC`

In [4]:
result = orders_with_customer_names(conn)

assert len(result) == 10
assert list(result.columns) == [
    "name", "order_date", "amount"
]
assert result.isna().sum().sum() == 0

print("Problem 2 completed successfully!")
print("Rows returned:", len(result))

display(result)

Problem 2 completed successfully!
Rows returned: 10


,name,order_date,amount
0,Kim Minji,2026-01-26,439700.0
1,Kim Minji,2026-01-24,257100.0
2,Kim Minji,2026-03-10,141000.0
3,Kim Minji,2026-03-15,250200.0
4,Kim Minji,2026-06-07,258700.0
5,Park Jisoo,2026-04-03,59800.0
6,Park Jisoo,2026-04-07,177000.0
7,Park Jisoo,2026-02-02,260500.0
8,Park Jisoo,2026-04-16,230900.0
9,Park Jisoo,2026-08-07,353100.0


### Problem 4 — Top N Customers by Spend
Return the top `n` customers by total spend — same idea as Problem 3, but capped to a specific count regardless of threshold.

**Hint:** `LIMIT`

In [5]:

def top_n_customers(conn, n=3):
    """
    Return the top n customers by total spending.

    Output columns:
    name, total_spent
    """

    query = """
        SELECT
            c.name,
            SUM(o.amount) AS total_spent
        FROM customers AS c
        INNER JOIN orders AS o
            ON c.customer_id = o.customer_id
        GROUP BY c.customer_id, c.name
        ORDER BY total_spent DESC
        LIMIT ?
    """

    return pd.read_sql(
        query,
        conn,
        params=(n,)
    )


# Assignment test
print(top_n_customers(conn, 3))


          name  total_spent
0   Jang Yerin    2167100.0
1  Yoon Doyoon    1609600.0
2   Baek Narae    1559500.0


In [6]:
result = top_n_customers(conn, 3)

assert len(result) == 3
assert list(result.columns) == ["name", "total_spent"]
assert result["total_spent"].is_monotonic_decreasing

print("Problem 4 completed successfully!")
print("Customers returned:", len(result))

display(result)

Problem 4 completed successfully!
Customers returned: 3


,name,total_spent
0,Jang Yerin,2167100.0
1,Yoon Doyoon,1609600.0
2,Baek Narae,1559500.0


In [7]:
print("Top 5 customers:")
display(top_n_customers(conn, 5))

print("Top 10 customers:")
display(top_n_customers(conn, 10))

Top 5 customers:


,name,total_spent
0,Jang Yerin,2167100.0
1,Yoon Doyoon,1609600.0
2,Baek Narae,1559500.0
3,Park Jisoo,1523300.0
4,Oh Jiwoo,1437300.0


Top 10 customers:


,name,total_spent
0,Jang Yerin,2167100.0
1,Yoon Doyoon,1609600.0
2,Baek Narae,1559500.0
3,Park Jisoo,1523300.0
4,Oh Jiwoo,1437300.0
5,Song Junho,1359000.0
6,Kim Minji,1346700.0
7,Ahn Jiho,1129700.0
8,Moon Hana,895700.0
9,Jung Hyunwoo,861200.0


---
## Part B — Python + SQL Safely

### Problem 5 — Search Orders by Customer Name (Safely!)
Write a function that looks up every order placed by a given customer name. **This must use a `?` placeholder** — it will be checking user-supplied input, exactly the SQL-injection scenario from the demo.

**Hint:** `cursor.execute(sql, (value,))`, then `.fetchall()`

In [8]:

def find_orders_by_customer(conn, customer_name):
    """
    Find all orders placed by a given customer.

    Returns:
        list of tuples (order_id, order_date, amount)
    """

    query = """
        SELECT
            o.order_id,
            o.order_date,
            o.amount
        FROM orders AS o
        INNER JOIN customers AS c
            ON o.customer_id = c.customer_id
        WHERE c.name = ?
        ORDER BY o.order_id
    """

    cursor = conn.cursor()
    cursor.execute(query, (customer_name,))

    return cursor.fetchall()


# Assignment test
print(find_orders_by_customer(conn, "Kim Minji"))


[(1, '2026-01-26', 439700.0), (2, '2026-01-24', 257100.0), (3, '2026-03-10', 141000.0), (4, '2026-03-15', 250200.0), (5, '2026-06-07', 258700.0)]


In [9]:
orders_found = find_orders_by_customer(conn, "Kim Minji")

assert isinstance(orders_found, list)
assert all(len(order) == 3 for order in orders_found)

print("Problem 5 completed successfully!")
print("Orders found:", len(orders_found))

for order in orders_found:
    print(order)

Problem 5 completed successfully!
Orders found: 5
(1, '2026-01-26', 439700.0)
(2, '2026-01-24', 257100.0)
(3, '2026-03-10', 141000.0)
(4, '2026-03-15', 250200.0)
(5, '2026-06-07', 258700.0)


In [10]:
test_input = "' OR 1=1 --"

result = find_orders_by_customer(conn, test_input)

print("Results for test input:", result)

assert result == []
print("Parameterized query test passed!")

Results for test input: []
Parameterized query test passed!


### Problem 6 — Monthly Revenue Report
Using `pd.read_sql()`, return total revenue grouped by month (format `YYYY-MM`), ordered chronologically.

**Hint:** SQLite's `strftime('%Y-%m', order_date)` extracts the year-month from a date string.

In [11]:

def monthly_revenue(conn):
    """
    Return total revenue grouped by month.

    Returns:
        Pandas DataFrame with columns:
        month, revenue
    """

    query = """
        SELECT
            strftime('%Y-%m', order_date) AS month,
            SUM(amount) AS revenue
        FROM orders
        GROUP BY strftime('%Y-%m', order_date)
        ORDER BY month ASC
    """

    return pd.read_sql(query, conn)


# Assignment test
print(monthly_revenue(conn))


     month    revenue
0  2026-01  1605200.0
1  2026-02  1924900.0
2  2026-03  1836600.0
3  2026-04  2277600.0
4  2026-05  1005400.0
5  2026-06  2817100.0
6  2026-07  1715100.0
7  2026-08  1906900.0
8  2026-09  1097800.0


In [12]:
revenue_df = monthly_revenue(conn)

assert isinstance(revenue_df, pd.DataFrame)
assert list(revenue_df.columns) == ["month", "revenue"]
assert revenue_df["month"].is_monotonic_increasing
assert revenue_df["revenue"].notna().all()

print("Problem 6 completed successfully!")

display(revenue_df)

Problem 6 completed successfully!


,month,revenue
0,2026-01,1605200.0
1,2026-02,1924900.0
2,2026-03,1836600.0
3,2026-04,2277600.0
4,2026-05,1005400.0
5,2026-06,2817100.0
6,2026-07,1715100.0
7,2026-08,1906900.0
8,2026-09,1097800.0


In [13]:
monthly_total = revenue_df["revenue"].sum()

database_total = pd.read_sql(
    "SELECT SUM(amount) AS total FROM orders",
    conn
)["total"].iloc[0]

print(f"Monthly revenue total: ₩{monthly_total:,.0f}")
print(f"Database revenue total: ₩{database_total:,.0f}")

assert abs(monthly_total - database_total) < 0.01

print("Revenue totals match!")

Monthly revenue total: ₩16,186,600
Database revenue total: ₩16,186,600
Revenue totals match!


---
## Part C — REST APIs

### Problem 7 — Fetch Weather for a City
Write a function that calls the Open-Meteo API (same one from the demo) for a given latitude/longitude and returns the parsed JSON `current_weather` dictionary. Check `response.status_code` and return `None` if the request fails.

**Hint:** `requests.get(url, params=...)`, `response.json()`

In [14]:

def get_current_weather(latitude, longitude):
    """
    Fetch current weather from Open-Meteo.

    Returns:
        dict containing current weather data,
        or None if the request fails.
    """

    url = "https://api.open-meteo.com/v1/forecast"

    params = {
        "latitude": latitude,
        "longitude": longitude,
        "current_weather": True
    }

    try:
        response = requests.get(
            url,
            params=params,
            timeout=10
        )

        if response.status_code != 200:
            print("API error:", response.status_code)
            return None

        data = response.json()

        return data.get("current_weather")

    except (requests.exceptions.RequestException, ValueError) as error:
        print("Request failed:", error)
        return None


# Assignment test — Seoul
print(get_current_weather(37.5665, 126.9780))


{'time': '2026-10-09T14:45', 'interval': 900, 'temperature': 16.3, 'windspeed': 1.6, 'winddirection': 297, 'is_day': 0, 'weathercode': 0}


In [15]:
weather = get_current_weather(37.5665, 126.9780)

if weather is not None:
    assert isinstance(weather, dict)
    assert "temperature" in weather

    print("Problem 7 completed successfully!")
    print("Seoul temperature:", weather["temperature"], "°C")
else:
    print("Live weather unavailable. Check your connection.")

Problem 7 completed successfully!
Seoul temperature: 16.3 °C


### Problem 8 — Weather for Multiple Cities into a DataFrame
Using the function from Problem 7, fetch the current weather for a list of `(city_name, lat, lon)` tuples and combine the results into a single DataFrame with a `city` column.

**Hint:** Build a list of dicts (adding a `"city"` key to each result), then `pd.DataFrame(list_of_dicts)`.

In [16]:

def weather_for_cities(cities):
    """
    Fetch weather for multiple cities.

    cities:
        List of (city_name, latitude, longitude)

    Returns:
        Pandas DataFrame containing city
        and current weather information.
    """

    records = []

    for city_name, latitude, longitude in cities:

        weather = get_current_weather(
            latitude,
            longitude
        )

        if weather is not None:
            record = {
                "city": city_name,
                **weather
            }

            records.append(record)

        else:
            print("Skipping failed request:", city_name)

    if not records:
        return pd.DataFrame(columns=["city"])

    return pd.DataFrame(records)


# Assignment test
test_cities = [
    ("Seoul", 37.5665, 126.9780),
    ("Busan", 35.1796, 129.0756),
    ("Tokyo", 35.6762, 139.6503)
]

print(weather_for_cities(test_cities))


    city              time  interval  temperature  windspeed  winddirection  \
0  Seoul  2026-10-09T14:45       900         16.3        1.6            297   
1  Busan  2026-10-09T14:45       900         19.8        4.5             61   
2  Tokyo  2026-10-09T14:45       900         17.1        3.3            347   

   is_day  weathercode  
0       0            0  
1       0            1  
2       0            0  


In [17]:
weather_df = weather_for_cities(test_cities)

print("Rows:", len(weather_df))
print("Columns:", weather_df.columns.tolist())

assert isinstance(weather_df, pd.DataFrame)
assert "city" in weather_df.columns

if len(weather_df) == 3:
    assert set(weather_df["city"]) == {
        "Seoul", "Busan", "Tokyo"
    }
    print("Problem 8 completed successfully!")
else:
    print("Some weather requests did not succeed.")

display(weather_df)

Rows: 3
Columns: ['city', 'time', 'interval', 'temperature', 'windspeed', 'winddirection', 'is_day', 'weathercode']
Problem 8 completed successfully!


,city,time,interval,temperature,windspeed,winddirection,is_day,weathercode
0,Seoul,2026-10-09T14:45,900,16.3,1.6,297,0,0
1,Busan,2026-10-09T14:45,900,19.8,4.5,61,0,1
2,Tokyo,2026-10-09T14:45,900,17.1,3.3,347,0,0


### Problem 9 — Respect the Rate Limit
Modify your approach from Problem 8 so that your function pauses `delay_seconds` between each API call, exactly as discussed in the demo's "Authentication and Rate Limits" section.

**Hint:** `time.sleep(delay_seconds)` inside your loop, after each request.

In [18]:

def weather_for_cities_paced(cities, delay_seconds=1):
    """
    Fetch weather for multiple cities while
    respecting API request pacing.

    Returns:
        Pandas DataFrame with city and weather fields.
    """

    records = []

    for city_name, latitude, longitude in cities:

        weather = get_current_weather(
            latitude,
            longitude
        )

        if weather is not None:
            records.append({
                "city": city_name,
                **weather
            })
        else:
            print("Weather unavailable for:", city_name)

        # Pause after each API request
        time.sleep(delay_seconds)

    if not records:
        return pd.DataFrame(columns=["city"])

    return pd.DataFrame(records)


# Assignment test
print(
    weather_for_cities_paced(
        test_cities,
        delay_seconds=1
    )
)


    city              time  interval  temperature  windspeed  winddirection  \
0  Seoul  2026-10-09T14:45       900         16.3        1.6            297   
1  Busan  2026-10-09T14:45       900         19.8        4.5             61   
2  Tokyo  2026-10-09T14:45       900         17.1        3.3            347   

   is_day  weathercode  
0       0            0  
1       0            1  
2       0            0  


In [19]:
paced_weather_df = weather_for_cities_paced(
    test_cities,
    delay_seconds=1
)

assert isinstance(paced_weather_df, pd.DataFrame)
assert "city" in paced_weather_df.columns

print("Problem 9 function executed successfully.")
print("Cities with retrieved weather:", len(paced_weather_df))

display(paced_weather_df)

Problem 9 function executed successfully.
Cities with retrieved weather: 3


,city,time,interval,temperature,windspeed,winddirection,is_day,weathercode
0,Seoul,2026-10-09T14:45,900,16.3,1.6,297,0,0
1,Busan,2026-10-09T14:45,900,19.8,4.5,61,0,1
2,Tokyo,2026-10-09T14:45,900,17.1,3.3,347,0,0


---
## Stretch Challenge (Optional) — Ethical Web Scraping

Scrape the first page of [books.toscrape.com](http://books.toscrape.com) and return a DataFrame with each book's title and price. Follow the demo's ethics checklist: use a descriptive `User-Agent`, and call `time.sleep(1)` after your request.

**Hint:** `BeautifulSoup`, `soup.select("article.product_pod")`, then pull the title from `h3 a` (the `title` attribute) and the price from `p.price_color` within each matched element.


In [20]:
%pip install beautifulsoup4

Note: you may need to restart the kernel to use updated packages.


In [21]:
import requests
import pandas as pd
import time
from bs4 import BeautifulSoup

print("All scraping libraries imported successfully!")

All scraping libraries imported successfully!


In [22]:

import requests
import pandas as pd
import time
from bs4 import BeautifulSoup


def scrape_book_titles_and_prices():
    """
    Scrape titles and prices from Books to Scrape.

    Returns:
        pandas DataFrame with title and price columns.
    """

    url = "https://books.toscrape.com/"

    headers = {
        "User-Agent": "KDU-DataScience-Student"
    }

    try:
        response = requests.get(
            url,
            headers=headers,
            timeout=10
        )

        time.sleep(1)
        response.raise_for_status()

        soup = BeautifulSoup(
            response.text,
            "html.parser"
        )

        books = []

        for article in soup.select("article.product_pod"):
            title_element = article.select_one("h3 a")
            price_element = article.select_one("p.price_color")

            if title_element is None or price_element is None:
                continue

            books.append({
                "title": title_element.get("title"),
                "price": price_element.get_text(strip=True)
            })

        return pd.DataFrame(
            books,
            columns=["title", "price"]
        )

    except requests.exceptions.RequestException as error:
        print("Scraping request failed:", error)
        return pd.DataFrame(columns=["title", "price"])


# Assignment test
books_df = scrape_book_titles_and_prices()

print("Books collected:", len(books_df))
display(books_df.head(10))


Books collected: 20


,title,price
0,A Light in the Attic,Â£51.77
1,Tipping the Velvet,Â£53.74
2,Soumission,Â£50.10
3,Sharp Objects,Â£47.82
4,Sapiens: A Brief History of Humankind,Â£54.23
5,The Requiem Red,Â£22.65
6,The Dirty Little Secrets of Getting Your Dream...,Â£33.34
7,The Coming Woman: A Novel Based on the Life of...,Â£17.93
8,The Boys in the Boat: Nine Americans and Their...,Â£22.60
9,The Black Maria,Â£52.15


In [23]:
assert not books_df.empty, "No books were retrieved."
assert list(books_df.columns) == ["title", "price"]
assert books_df["title"].notna().all()

print("Stretch Challenge completed successfully!")

Stretch Challenge completed successfully!


---
## Cleanup
Run this last to close your database connection.

In [24]:

# Final Cleanup — Close SQLite database connection

if "conn" in globals():
    conn.close()
    print("Database connection closed successfully.")
else:
    print("No active database connection was created in this session.")

print("Cleanup completed.")


Database connection closed successfully.
Cleanup completed.
